# Block 6. Past fire cases: Valparaíso, February 2024

Plan: `docs/eda-plan.md`, block 6, limited to Valparaíso 2024 (`docs/work-split.md`, module M5).
Findings: `docs/findings/06-valparaiso-2024.md`.

Questions:

1. When did each satellite first see the fire, against the reported start (about 12:05 civil time, 2 February 2024)?
2. How do MODIS and VIIRS points compare for the same event?
3. Does February 2024 stand out in the harmonized burn calendar of the area and the region?
4. What did the risk layer say for that week, if `scripts/risk_baseline.py` or `scripts/risk_model.py` was run?

Needs: the hotspot table with time fields (`scripts/build_hotspots.py`) and the harmonized dataset (`scripts/harmonize.py`).

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "NASA_Data_Sources" / "processed"
HOTSPOTS = DATA / "hotspots"
HARMONIZED = DATA / "harmonized" / "cells_daily.parquet"
RISK = DATA / "risk"
FIG_DIR = ROOT / "docs" / "findings"

# Case definition. Reported start: CONAF brigades at Complejo Las Tablas from 12:05, first sighting
# between 12:00 and 13:00 (sources in docs/findings/06-valparaiso-2024.md).
CASE = "Valparaíso 2024"
TZ = "America/Santiago"
REPORTED_START = pd.Timestamp("2024-02-02 12:05", tz=TZ)
BBOX = {"lat_min": -33.25, "lat_max": -32.95, "lon_min": -71.75, "lon_max": -71.30}
WINDOW = (pd.Timestamp("2024-01-30").date(), pd.Timestamp("2024-02-10").date())  # burn_date
CELL_DEG = 0.25

## 1. Hotspots of the event

In [ ]:
cols = ["acq_datetime", "burn_date", "burn_year", "burn_month", "local_wall_time", "pass_slot",
        "solar_time_h", "latitude", "longitude", "admin1", "instrument", "satellite", "processing",
        "confidence_level", "frp_mw", "fire_type"]
filters = [("year", "in", [2024]),
           ("latitude", ">=", BBOX["lat_min"]), ("latitude", "<=", BBOX["lat_max"]),
           ("longitude", ">=", BBOX["lon_min"]), ("longitude", "<=", BBOX["lon_max"])]
hs = pd.read_parquet(HOTSPOTS, columns=cols, filters=filters)
hs["burn_date"] = pd.to_datetime(hs["burn_date"]).dt.date
hs = hs[(hs["burn_date"] >= WINDOW[0]) & (hs["burn_date"] <= WINDOW[1])]
hs = hs[~hs["fire_type"].astype("Int8").isin([1, 2, 3]).fillna(False).to_numpy()].copy()
hs["satellite"] = hs["satellite"].astype(str)
hs["instrument"] = hs["instrument"].astype(str)
# Civil time from the UTC instant (rule 6: always UTC -> local).
hs["civil_time"] = pd.to_datetime(hs["acq_datetime"], utc=True).dt.tz_convert(TZ)
print(f"{len(hs):,} hotspots in the box between burn_date {WINDOW[0]} and {WINDOW[1]}")
hs.groupby(["burn_date", "instrument"]).size().unstack(fill_value=0)

## 2. Timeline per satellite and pass

In [ ]:
timeline = (hs.groupby(["satellite", "burn_date", "pass_slot"], observed=True)
              .agg(points=("frp_mw", "size"), frp_sum_mw=("frp_mw", "sum"),
                   first_civil=("civil_time", "min"), last_civil=("civil_time", "max"))
              .reset_index().sort_values("first_civil"))
timeline["first_civil"] = timeline["first_civil"].dt.strftime("%Y-%m-%d %H:%M")
timeline["last_civil"] = timeline["last_civil"].dt.strftime("%H:%M")
timeline

## 3. First detection against the reported start

Polar-orbiting satellites pass over a place a few times a day: Terra in the morning, Aqua and the
VIIRS satellites in the early afternoon, and again at night. A fire that starts around noon can only
be seen at the next pass.

In [ ]:
after = hs[hs["civil_time"] >= REPORTED_START - pd.Timedelta(hours=1)]
first = (after.sort_values("civil_time").groupby("satellite").first()[["civil_time", "pass_slot", "frp_mw"]])
first["delay_h"] = ((first["civil_time"] - REPORTED_START).dt.total_seconds() / 3600).round(2)
first["civil_time"] = first["civil_time"].dt.strftime("%Y-%m-%d %H:%M")
first = first.sort_values("delay_h")
print(f"Reported start: {REPORTED_START:%Y-%m-%d %H:%M} ({TZ})")
first

## 4. MODIS against VIIRS for the same event

In [ ]:
core = hs[hs["burn_date"].isin([pd.Timestamp("2024-02-02").date(), pd.Timestamp("2024-02-03").date()])]
per_sat = core.groupby(["satellite", "pass_slot"], observed=True).size().unstack(fill_value=0)
per_sat["total"] = per_sat.sum(axis=1)
display(per_sat)
modis = core[core["instrument"] == "MODIS"]
snpp = core[core["satellite"] == "Suomi-NPP"]
if len(modis):
    print(f"Suomi-NPP / MODIS points on 2-3 February: {len(snpp) / len(modis):.2f}")
    aqua = core[core["satellite"] == "Aqua"]
    if len(aqua):
        print(f"Suomi-NPP / Aqua points: {len(snpp) / len(aqua):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, (name, sub) in zip(axes, [("MODIS (Terra + Aqua)", core[core["instrument"] == "MODIS"]),
                                  ("VIIRS (all satellites)", core[core["instrument"] == "VIIRS"])]):
    hours = (sub["civil_time"] - REPORTED_START).dt.total_seconds() / 3600
    sc = ax.scatter(sub["longitude"], sub["latitude"], c=hours, s=6, cmap="viridis", vmin=0, vmax=40)
    ax.set_title(f"{name}: {len(sub):,} points")
    ax.set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
fig.colorbar(sc, ax=axes, label="Hours after the reported start")
fig.suptitle(f"{CASE}: hotspots on burn_date 2 and 3 February 2024")
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_DIR / "06-valparaiso-hotspots.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. Does February 2024 stand out in the burn calendar?

Harmonized counts (VIIRS-equivalent, `scripts/harmonize.py`). Always sum `hotspots_harmonized`.

In [ ]:
c2 = pd.read_parquet(HARMONIZED)
c2["burn_date"] = pd.to_datetime(c2["burn_date"])
c2["year"], c2["month"] = c2["burn_date"].dt.year, c2["burn_date"].dt.month
in_box = ((c2["cell_lat"] + CELL_DEG > BBOX["lat_min"]) & (c2["cell_lat"] < BBOX["lat_max"]) &
          (c2["cell_lon"] + CELL_DEG > BBOX["lon_min"]) & (c2["cell_lon"] < BBOX["lon_max"]))
region = c2["admin1"].astype(str).str.contains("Valpara", na=False)

def february_rank(mask, label):
    feb = c2[mask & (c2["month"] == 2)].groupby("year")["hotspots_harmonized"].sum()
    feb = feb.reindex(range(2001, 2027), fill_value=0.0)
    past = feb.loc[2001:2023]
    value = feb.loc[2024]
    out = {"area": label, "feb_2024": round(value), "median_feb_2001_2023": round(past.median()),
           "ratio_to_median": round(value / past.median(), 1) if past.median() else np.nan,
           "rank_among_febs": int((feb > value).sum() + 1), "years": len(feb)}
    return out, feb

rows, series = [], {}
for mask, label in [(in_box, "Case box (cells)"), (region, "Valparaíso region")]:
    out, feb = february_rank(mask, label)
    rows.append(out)
    series[label] = feb
display(pd.DataFrame(rows))

fig, ax = plt.subplots(figsize=(11, 3.5))
s = series["Case box (cells)"]
ax.bar(s.index, s.values, color=["#c0392b" if y == 2024 else "#7f8c8d" for y in s.index])
ax.set_title("Harmonized hotspots in February, case box (2024 in red)")
ax.set_ylabel("VIIRS-equivalent hotspots")
fig.savefig(FIG_DIR / "06-valparaiso-february.png", dpi=120, bbox_inches="tight")
plt.show()

## 6. What did the risk layer say?

Runs only if the risk outputs exist. The week of 2 February 2024 is in the test years of the model
(2023-2025), so its probability is out of sample.

In [ ]:
eval_file = next((RISK / f for f in ["model_eval.parquet", "baseline_eval.parquet"]
                  if (RISK / f).exists()), None)
if eval_file is None:
    print("No risk outputs yet: run scripts/risk_baseline.py or scripts/risk_model.py")
else:
    risk = pd.read_parquet(eval_file)
    risk["week_start"] = pd.to_datetime(risk["week_start"])
    wk = risk[(risk["week_start"] <= "2024-02-02") & (risk["week_start"] > pd.Timestamp("2024-02-02") - pd.Timedelta(days=8))]
    wk = wk.assign(pct_rank=wk["probability"].rank(pct=True))
    box = wk[(wk["cell_lat"] + CELL_DEG > BBOX["lat_min"]) & (wk["cell_lat"] < BBOX["lat_max"]) &
             (wk["cell_lon"] + CELL_DEG > BBOX["lon_min"]) & (wk["cell_lon"] < BBOX["lon_max"])]
    print(f"{eval_file.name}, week starting {wk['week_start'].iloc[0]:%Y-%m-%d}: "
          f"{len(wk):,} cells, {wk['observed'].mean():.1%} had fire")
    display(box[["cell_lat", "cell_lon", "baseline_probability", "probability", "pct_rank", "observed"]]
            .sort_values("probability", ascending=False))

## 7. Results to copy into `docs/findings/06-valparaiso-2024.md`

Fill the findings file with the numbers of sections 3 to 6: first detection and delay per satellite,
Suomi-NPP / MODIS ratio for the event, February 2024 against the median, and the risk percentile.